In [1]:
import pandas as pd
import os

In [2]:
def get_info_from_GO_Term(GO_Term):
    definition =  GO_Term[GO_Term.index("\ndef")+6:]
    definition = definition[:definition.index("\n")]
    
    name = GO_Term[GO_Term.index("\nname")+7:]
    name = name[:name.index("\n")]
    
    ID = GO_Term[GO_Term.index("\nid")+5:]
    ID = ID[:ID.index("\n")]
    return(definition, name, ID)

In [3]:
import numpy as np

def get_RHEA_reaction_IDs(GO_Term):
    RHEA = np.nan
    if "xref: RHEA:" in GO_Term:
        RHEA = GO_Term[GO_Term.find("xref: RHEA:") + len("xref: RHEA:"):]
        RHEA = RHEA[:RHEA.find("\n")]
    return(RHEA)

In [4]:
df = pd.DataFrame(columns = ["GO ID", "Definition", "Name", "RHEA ID"])

#read go obo file
path_go_obo = os.path.join("..",'go.obo')
file = open(path_go_obo, 'r')
Lines = file.readlines()

term = ""
start = False

#process data from go.obo-file and save it in a pandas DataFrame:
for line in Lines:
    if '[Term]\n' in line and not start:
        start = True
        term = line
    elif '[Term]\n' in line: 
        definition, name, ID = get_info_from_GO_Term(GO_Term = term)
        if "Catalysis of the reaction" in definition:
            RHEA_ID= get_RHEA_reaction_IDs(GO_Term = term)
            df = pd.concat((df, pd.DataFrame({"GO ID" : [ID] , "Definition" : [definition], "Name" : [name],
                            "RHEA ID" : [RHEA_ID]})), ignore_index = True)
        term = line
    elif start:
        term = term + line
    
df.head()

,GO ID,Definition,Name,RHEA ID
0,GO:0000010,"""Catalysis of the reaction: (2E,6E)-farnesyl d...",heptaprenyl diphosphate synthase activity,27794
1,GO:0000016,"""Catalysis of the reaction: lactose + H2O = D-...",lactase activity,10076
2,GO:0000034,"""Catalysis of the reaction: adenine + H2O = hy...",adenine deaminase activity,23688
3,GO:0000048,"""Catalysis of the reaction: peptidyl-tRNA(1) +...",peptidyltransferase activity,NaN
4,GO:0000104,"""Catalysis of the reaction: succinate + accept...",succinate dehydrogenase activity,16357


In [5]:
digits = [str(i) + " " for i in range(1,100)] + ["a "]
def process_metabolites(metabolites_list):
    for i in range(len(metabolites_list)):
        met = metabolites_list[i]
        if met[0:2] in digits:
            met = met[2:]
        elif met[0:3] in digits:
            met = met[3:]
            
        if met[-1] == "(":      
            met = met[:-1]
        
        metabolites_list[i] = met
    return(metabolites_list)


def find_substrates(definition):
    starter = ["OBSOLETE. Catalysis of the reaction: ", 
           "OBSOLETE. Catalysis of the reactions: ",
           "OBSOLETE. Catalysis of the reaction:", 
           "OBSOLETE. Catalysis of the reaction ",
           "Catalysis of the reaction: ", 
           "Catalysis of the reactions: ",
           "Catalysis of the reaction:", 
           "Catalysis of the reaction "]
    substrates = []
    
    #trying to find one of the "starters" until successfull or until all startes have been tried:
    i = 0
    successfull = False
    while not successfull and i < len(starter):
        start = starter[i]
        try: 
            definition.index(start)
            definition = definition[len(start)+1:definition.index('."')]
            if "<=>" in definition:
                substrates = definition.split(" <=> ")[0]
            elif "->" in definition:
                substrates = definition.split(" -> ")[0]
            elif "=>" in definition:
                substrates = definition.split(" => ")[0]
            elif "= " in definition:
                substrates = definition.split("= ")[0]
            elif "=" in definition:
                substrates = definition.split("=")[0]
            else:
                print("Could not find substrate in the following definition: %s" % definition)
            substrates = substrates.replace(" + ", ";")
            substrates = substrates.split(";")
            substrates = process_metabolites(metabolites_list = substrates)
            successfull = True
        except:
            pass
        i = i+1
        
    return(substrates)

In [6]:
df["substrates"] = ""

for ind in df.index:
    df["substrates"][ind] = find_substrates(definition = df["Definition"][ind])
    
df

Could not find substrate in the following definition: which results in unsaturation at C-7 in the B ring of sterols
Could not find substrate in the following definition: GlcNAc-1,6-anhMurNAc-L-Ala-gamma-D-Glu-DAP-D-Ala + H2O glcNAc-1,6-anhMurNAc + L-Ala-gamma-D-Glu-DAP-D-Ala
Could not find substrate in the following definition: involving the transfer of a phosphatidate (otherwise known as diacylglycerol 3-phosphosphate) group
Could not find substrate in the following definition: ATP + undecaprenol + all-trans-undecaprenyl phosphate + ADP + H+
Could not find substrate in the following definition: 7-hydroxymethyl chlorophyll a + 2 reduced ferredoxin + 2 H+ chlorophyll a + 2 oxidized ferredoxin + H2O


,GO ID,Definition,Name,RHEA ID,substrates
0,GO:0000010,"""Catalysis of the reaction: (2E,6E)-farnesyl d...",heptaprenyl diphosphate synthase activity,27794,"[(2E,6E)-farnesyl diphosphate, isopentenyl dip..."
1,GO:0000016,"""Catalysis of the reaction: lactose + H2O = D-...",lactase activity,10076,"[lactose, H2O ]"
2,GO:0000034,"""Catalysis of the reaction: adenine + H2O = hy...",adenine deaminase activity,23688,"[adenine, H2O ]"
3,GO:0000048,"""Catalysis of the reaction: peptidyl-tRNA(1) +...",peptidyltransferase activity,NaN,"[peptidyl-tRNA(1), aminoacyl-tRNA(2) ]"
4,GO:0000104,"""Catalysis of the reaction: succinate + accept...",succinate dehydrogenase activity,16357,"[succinate, acceptor ]"
...,...,...,...,...,...
6704,GO:1990833,"""Catalysis of the reaction: ATP + H2O = ADP + ...",clathrin-uncoating ATPase activity,NaN,"[ATP, H2O ]"
6705,GO:1990883,"""Catalysis of the reaction: acetyl-CoA + cytid...",rRNA cytidine N-acetyltransferase activity,NaN,"[acetyl-CoA, cytidine ]"
6706,GO:1990887,"""OBSOLETE. Catalysis of the reaction: 2-polypr...",obsolete 2-polyprenyl-3-methyl-5-hydroxy-6-met...,NaN,"[2-polyprenyl-3-methyl-5-hydroxy-6-methoxy-1,4..."
6707,GO:1990888,"""Catalysis of the reaction: 2-polyprenyl-6-hyd...",2-polyprenyl-6-hydroxyphenol O-methyltransfera...,NaN,"[2-polyprenyl-6-hydroxyphenol, S-adenosyl-L-me..."


In [7]:
def extract_RHEA_ID_and_CHEBI_IDs(entry):
    RHEA_ID = entry[0][len("ENTRY"): -1]
    RHEA_ID = RHEA_ID.split(" ")[-1]
    CHEBI_IDs = entry[2][len("EQUATION"): -1]
    CHEBI_IDs = CHEBI_IDs[CHEBI_IDs.index("CHEBI"):]
    return(RHEA_ID, CHEBI_IDs)

In [8]:
def get_substrate_IDs(IDs):
    IDs = IDs.split(" = ")[0]
    IDs = IDs.split(" => ")[0]
    IDs = IDs.split(" <=> ")[0]
    IDs = IDs.replace(" + ", ";")
    IDs = IDs.split(";")
    return([ID.split(" ")[-1] for ID in IDs])

In [9]:
df_RHEA = pd.DataFrame(columns = ["RHEA ID", "CHEBI IDs", "CHEBI_ID_list"])

file1 = open(os.path.join(".." ,"rhea-reactions.txt"), 'r')
lines = file1.readlines()
entries = []
entry = []
from tqdm import tqdm
for line in lines:
    if '///\n' in line:
        entry.append(line)
        RHEA_ID, CHEBI_IDs = extract_RHEA_ID_and_CHEBI_IDs(entry)
        CHEBI_ID_list = get_substrate_IDs(IDs = CHEBI_IDs)
        # df_RHEA = pd.concat((df_RHEA, pd.DataFrame({"RHEA ID" : [RHEA_ID], "CHEBI_ID_list" : [CHEBI_ID_list]})), ignore_index = True)
        entries.append(entry)
        entry=[]
    else:
        entry.append(line)
        
# df_RHEA["RHEA ID"] = [float(ID.split(":")[-1]) for ID in df_RHEA["RHEA ID"]]
# df_RHEA

In [10]:
from tqdm import tqdm
import contextlib
import joblib

@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """Context manager to patch joblib to report into tqdm progress bar given as argument"""

    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()

In [11]:
def get_substrates(entry):
    RHEA_ID, CHEBI_IDs = extract_RHEA_ID_and_CHEBI_IDs(entry)
    CHEBI_ID_list = get_substrate_IDs(IDs = CHEBI_IDs)
    return pd.DataFrame({"RHEA ID" : [RHEA_ID], "CHEBI_ID_list" : [CHEBI_ID_list]})

In [12]:
from joblib import Parallel, delayed

parallel_callback = Parallel(n_jobs=-1, backend="multiprocessing", prefer="threads")
with tqdm_joblib(tqdm(desc="get substrates", total=len(entries))):
    res = parallel_callback(
        delayed(get_substrates)(entry)
        for entry in entries)

get substrates: 100%|██████████| 65640/65640 [00:08<00:00, 7323.54it/s] 


In [13]:
df_RHEA = pd.concat(res, ignore_index = True)
df_RHEA["RHEA ID"] = [float(ID.split(":")[-1]) for ID in df_RHEA["RHEA ID"]]

In [14]:
df["RHEA ID"] = [str(ID) for ID in df["RHEA ID"]]

In [15]:
df["RHEA ID"] = [float(ID.split(" ")[0]) for ID in df["RHEA ID"]]
df_catalytic = df.merge(df_RHEA, how = "left", on = ["RHEA ID"])
df_catalytic.head()

,GO ID,Definition,Name,RHEA ID,substrates,CHEBI_ID_list
0,GO:0000010,"""Catalysis of the reaction: (2E,6E)-farnesyl d...",heptaprenyl diphosphate synthase activity,27794.0,"[(2E,6E)-farnesyl diphosphate, isopentenyl dip...","[CHEBI:175763, CHEBI:128769]"
1,GO:0000016,"""Catalysis of the reaction: lactose + H2O = D-...",lactase activity,10076.0,"[lactose, H2O ]","[CHEBI:15377, CHEBI:17716]"
2,GO:0000034,"""Catalysis of the reaction: adenine + H2O = hy...",adenine deaminase activity,23688.0,"[adenine, H2O ]","[CHEBI:16708, CHEBI:15378, CHEBI:15377]"
3,GO:0000048,"""Catalysis of the reaction: peptidyl-tRNA(1) +...",peptidyltransferase activity,NaN,"[peptidyl-tRNA(1), aminoacyl-tRNA(2) ]",NaN
4,GO:0000104,"""Catalysis of the reaction: succinate + accept...",succinate dehydrogenase activity,16357.0,"[succinate, acceptor ]","[CHEBI:13193, CHEBI:30031]"


In [16]:
df_substrates = pd.DataFrame(columns = ["GO ID", "molecule", "molecule ID", "RHEA ID"])
df_catalytic["CHEBI_ID_list"].loc[pd.isnull(df_catalytic["CHEBI_ID_list"])] = ""

for ind in df_catalytic.index:
    GO_ID = df_catalytic["GO ID"][ind]
    RHEA_ID = df_catalytic["RHEA ID"][ind]
    
    if len(df_catalytic["CHEBI_ID_list"][ind]) > 0:
        IDs = df_catalytic["CHEBI_ID_list"][ind]
        for ID in IDs:
            df_substrates = pd.concat((df_substrates, pd.DataFrame({"GO ID" : [GO_ID], "molecule": [np.nan], "molecule ID" : [ID],
                                                 "RHEA ID" : [RHEA_ID]})),
                                                 ignore_index = True)
    else:
        metabolites = df_catalytic["substrates"][ind]
        for met in metabolites:
            df_substrates = pd.concat((df_substrates, pd.DataFrame({"GO ID" : [GO_ID], "molecule": [met], "molecule ID" : [np.nan],
                                                 "RHEA ID" : [RHEA_ID]})),
                                                 ignore_index = True)
            
df_substrates.reset_index(inplace = True, drop = True)
df_substrates

/tmp/ipykernel_1186082/3113526184.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_catalytic["CHEBI_ID_list"].loc[pd.isnull(df_catalytic["CHEBI_ID_list"])] = ""


,GO ID,molecule,molecule ID,RHEA ID
0,GO:0000010,NaN,CHEBI:175763,27794.0
1,GO:0000010,NaN,CHEBI:128769,27794.0
2,GO:0000016,NaN,CHEBI:15377,10076.0
3,GO:0000016,NaN,CHEBI:17716,10076.0
4,GO:0000034,NaN,CHEBI:16708,23688.0
...,...,...,...,...
14906,GO:1990887,S-adenosyl-L-methionine,NaN,NaN
14907,GO:1990888,2-polyprenyl-6-hydroxyphenol,NaN,NaN
14908,GO:1990888,S-adenosyl-L-methionine,NaN,NaN
14909,GO:1990965,cytosylglucuronic acid,NaN,NaN


In [17]:
metabolites = []
for ind in df_substrates.index:
    if pd.isnull(df_substrates["molecule ID"][ind]):
        metabolites = metabolites + [df_substrates["molecule"][ind]]
    
df_unmapped = pd.DataFrame(data = {"metabolites" : list(set(metabolites))})
df_unmapped

,metabolites
0,1-18:2-2-18:3-phosphatidylcholine
1,phosphate
2,D-fructose
3,5'-dephospho-RNA
4,protein-cysteine
...,...
2408,triethanolamine
2409,mucus glycoprotein
2410,alpha-pinene
2411,H-


In [18]:
CURRENT_DIR = os.getcwd()
import pickle

# read pickle

drugs_df = pd.read_pickle(os.path.join(CURRENT_DIR, "data", "KEGG_drugs_df.pkl"))
compounds_df = pd.read_pickle(os.path.join(CURRENT_DIR, "data", "KEGG_substrate_df.pkl"))
KEGG_substrate_df = compounds_df.append(drugs_df).reset_index(drop = True)

##If we have multiple IDs for the same substrate name, we keep the first ID:

substrate_list = KEGG_substrate_df["substrate"].tolist()
substrate_counts = KEGG_substrate_df["substrate"].value_counts()

# Create a mask for rows with duplicates
duplicate_mask = KEGG_substrate_df["substrate"].isin(substrate_counts[substrate_counts > 1].index)

# Filter out the duplicates
help_df = KEGG_substrate_df[duplicate_mask]

# Group by 'substrate' and get the indices of all but the first occurrence
droplist = help_df.groupby('substrate').apply(lambda x: x.index[1:]).explode().tolist()
 
KEGG_substrate_df.drop(droplist, inplace = True)

KEGG_substrate_df["substrate"] = [name.lower() for name in KEGG_substrate_df["substrate"]]
df_unmapped["substrate"] = [name.lower() for name in df_unmapped["metabolites"]]

df_unmapped = df_unmapped.merge(KEGG_substrate_df, on = "substrate", how = "left")
print("For %s out of %s data points we could not map the substrate name to a KEGG ID." %
      (sum(pd.isnull(df_unmapped["KEGG ID"])), len(df_unmapped)))

For 1576 out of 2413 data points we could not map the substrate name to a KEGG ID.


/tmp/ipykernel_1186082/3759740352.py:8: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  KEGG_substrate_df = compounds_df.append(drugs_df).reset_index(drop = True)


In [19]:
df_unmapped

,metabolites,substrate,KEGG ID
0,1-18:2-2-18:3-phosphatidylcholine,1-18:2-2-18:3-phosphatidylcholine,NaN
1,phosphate,phosphate,C00009
2,D-fructose,d-fructose,NaN
3,5'-dephospho-RNA,5'-dephospho-rna,NaN
4,protein-cysteine,protein-cysteine,NaN
...,...,...,...
2408,triethanolamine,triethanolamine,NaN
2409,mucus glycoprotein,mucus glycoprotein,NaN
2410,alpha-pinene,alpha-pinene,C09880
2411,H-,h-,NaN


In [20]:
matches= pd.read_pickle(os.path.join(CURRENT_DIR, "data", "Pubchem_substrate_matches_GO_Terms.pkl"))
matches["CID"] = [int(cid) if not pd.isnull(cid) else np.nan for cid in matches["CID"]]
matches.head()
matches = matches.loc[~pd.isnull(matches["CID"])]

#load the resulting file and store it in a DataFrame:
Pubchem_CID_to_KEGG_df = pd.read_csv(os.path.join(CURRENT_DIR, "data", "name_map_metaboanalyst.csv"), sep = ",")
#rename columns:
Pubchem_CID_to_KEGG_df.rename(columns = {"PubChem" : "CID"}, inplace = True)
Pubchem_CID_to_KEGG_df.drop(columns = ["Match","Query", "HMDB", "SMILES", "Comment"], inplace = True)

#merge this DataFrame with the DataFrame called matches:
macthes_with_KEGG_IDs = pd.merge(matches, Pubchem_CID_to_KEGG_df, how='left', on=['CID'])
macthes_with_KEGG_IDs.rename(columns = {"Match" : "substrate"}, inplace = True)
macthes_with_KEGG_IDs

,Metabolite,CID,ChEBI,KEGG,METLIN
0,coash,87642.0,15346.0,C00010,6235.0
1,dehydroabietadienal,11694869.0,52487.0,NaN,NaN
2,tricaffeoyl spermidine,15241070.0,NaN,NaN,NaN
3,chlorophyllide b(1-),86289090.0,NaN,NaN,NaN
4,beta-chaconine,119393.0,NaN,NaN,NaN
...,...,...,...,...,...
447,"2,4-dinitrocyclohexanone",9543365.0,NaN,NaN,NaN
448,2-(methylthio)benzothiazole,11989.0,NaN,NaN,NaN
449,indole acetic acid,802.0,16411.0,C00954,5211.0
450,(r)-3-hydroxyicosanoyl-coa(4-),72193813.0,NaN,NaN,NaN


In [21]:
df_unmapped["CHEBI ID"] = np.nan

for ind in df_unmapped.index:
    if pd.isnull(df_unmapped["KEGG ID"][ind]):
        substrate = df_unmapped["substrate"][ind]
        try:
            KEGG_ID = list(macthes_with_KEGG_IDs.loc[macthes_with_KEGG_IDs["substrate"]== substrate]["KEGG ID"])[0]
            df_unmapped["KEGG ID"][ind] = KEGG_ID
        except:
            None
        try:
            CHEBI_ID = list(macthes_with_KEGG_IDs.loc[macthes_with_KEGG_IDs["substrate"]== substrate]["ChEBI"])[0]
            df_unmapped["CHEBI ID"][ind] = CHEBI_ID
        except:
            None
df_unmapped

,metabolites,substrate,KEGG ID,CHEBI ID
0,1-18:2-2-18:3-phosphatidylcholine,1-18:2-2-18:3-phosphatidylcholine,NaN,NaN
1,phosphate,phosphate,C00009,NaN
2,D-fructose,d-fructose,NaN,NaN
3,5'-dephospho-RNA,5'-dephospho-rna,NaN,NaN
4,protein-cysteine,protein-cysteine,NaN,NaN
...,...,...,...,...
2408,triethanolamine,triethanolamine,NaN,NaN
2409,mucus glycoprotein,mucus glycoprotein,NaN,NaN
2410,alpha-pinene,alpha-pinene,C09880,NaN
2411,H-,h-,NaN,NaN


In [22]:
for ind in df_substrates.index:
    if pd.isnull(df_substrates["molecule ID"][ind]):
        met = df_substrates["molecule"][ind]

        ID = list(df_unmapped["CHEBI ID"].loc[df_unmapped["metabolites"] == met])[0]
        if pd.isnull(ID):
            ID = list(df_unmapped["KEGG ID"].loc[df_unmapped["metabolites"] == met])[0]

        df_substrates["molecule ID"][ind] = ID
df_substrates

/tmp/ipykernel_1186082/750475231.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_substrates["molecule ID"][ind] = ID


,GO ID,molecule,molecule ID,RHEA ID
0,GO:0000010,NaN,CHEBI:175763,27794.0
1,GO:0000010,NaN,CHEBI:128769,27794.0
2,GO:0000016,NaN,CHEBI:15377,10076.0
3,GO:0000016,NaN,CHEBI:17716,10076.0
4,GO:0000034,NaN,CHEBI:16708,23688.0
...,...,...,...,...
14906,GO:1990887,S-adenosyl-L-methionine,NaN,NaN
14907,GO:1990888,2-polyprenyl-6-hydroxyphenol,C17551,NaN
14908,GO:1990888,S-adenosyl-L-methionine,NaN,NaN
14909,GO:1990965,cytosylglucuronic acid,NaN,NaN


In [23]:
df_substrates.drop(columns = ["molecule"], inplace= True)
df_substrates = df_substrates.loc[~pd.isnull(df_substrates["molecule ID"])]
df_substrates.reset_index(inplace = True, drop = True)
df_substrates

,GO ID,molecule ID,RHEA ID
0,GO:0000010,CHEBI:175763,27794.0
1,GO:0000010,CHEBI:128769,27794.0
2,GO:0000016,CHEBI:15377,10076.0
3,GO:0000016,CHEBI:17716,10076.0
4,GO:0000034,CHEBI:16708,23688.0
...,...,...,...
11445,GO:1990817,CHEBI:30616,11332.0
11446,GO:1990817,CHEBI:140395,11332.0
11447,GO:1990833,C00002,NaN
11448,GO:1990883,C00024,NaN


In [24]:
df_substrates.to_csv(os.path.join(CURRENT_DIR, "substrates.csv"), sep = ",", index = False)

In [25]:
len(df_catalytic) - len(set(df_substrates["GO ID"]))

870

In [26]:
catalytic_go_terms = list(set(df["GO ID"]))

In [27]:
len(catalytic_go_terms)

6709

In [28]:
ECO_to_GAF = pd.read_csv(os.path.join("data", 'ECO_to_GAF.tsv'), sep = "\t")
exp_evidence = ["EXP","IDA","IPI","IMP","IGI","IEP", "HTP","HDA","HMP","HGI","HEP"]
phylo_evidence = ["IBA","IBD","IKR","IRD"]

In [29]:
df_GO_UID = pd.read_pickle(os.path.join("data","experimental_phylogenetic_df_GO_UID.pkl"))

In [30]:
# df_UID_MID = pd.DataFrame(columns =["Uniprot ID", "molecule ID", "evidence", "RHEA ID"])

# for ind in df_GO_UID.index:
#     if ind >= -1:
#         GO_ID = df_GO_UID["GO Term"][ind]
#         try:
#             RHEA_ID = list(df_catalytic["RHEA ID"].loc[df_catalytic["GO ID"] == GO_ID])[0]
#         except:
#             RHEA_ID = np.nan
#             print(GO_ID)
#         UID = df_GO_UID["Uniprot ID"][ind]
#         evidence = df_GO_UID["evidence"][ind]
#         met_IDs = list(df_substrates["molecule ID"].loc[df_substrates["GO ID"] == GO_ID])
#         for met_ID in met_IDs:
#             df_UID_MID = df_UID_MID.append({"Uniprot ID" : UID, "molecule ID" : met_ID,
#                                            "evidence": evidence, "RHEA ID" : RHEA_ID}, ignore_index = True)
#     if ind % 1000 ==1:
#         print(ind)
        
# df_UID_MID.drop_duplicates(inplace = True)

# Uniprot_IDs = list(set(df_UID_MID["Uniprot ID"]))
# print(len(Uniprot_IDs), len(list(set(df_UID_MID["molecule ID"]))))

# df_UID_MID

In [38]:
import numpy as np
import pandas as pd

# Initialize a list to collect data
data = []

# Loop through each index in df_GO_UID
bar = tqdm(total = len(df_GO_UID))
for ind in df_GO_UID.index:
    bar.update(1)
    if ind >= -1:  # This condition seems redundant, as ind will always be >= -1 in a for loop
        GO_ID = df_GO_UID.at[ind, "GO Term"]

        # Try to get the RHEA ID, handle KeyError if the GO ID is not found
        try:
            RHEA_ID = df_catalytic.loc[df_catalytic["GO ID"] == GO_ID, "RHEA ID"].values[0]
        except IndexError:
            RHEA_ID = np.nan

        UID = df_GO_UID.at[ind, "Uniprot ID"]
        evidence = df_GO_UID.at[ind, "evidence"]
        met_IDs = df_substrates.loc[df_substrates["GO ID"] == GO_ID, "molecule ID"].tolist()

        # Collect data in the list
        for met_ID in met_IDs:
            data.append({"Uniprot ID": UID, "molecule ID": met_ID, "evidence": evidence, "RHEA ID": RHEA_ID})

# Create the DataFrame from the collected data
df_UID_MID = pd.DataFrame(data)

# Remove duplicates
df_UID_MID.drop_duplicates(inplace=True)

# Get unique Uniprot IDs and molecule IDs
Uniprot_IDs = df_UID_MID["Uniprot ID"].unique()
molecule_IDs = df_UID_MID["molecule ID"].unique()

# Print the lengths of unique Uniprot IDs and molecule IDs
print(len(Uniprot_IDs), len(molecule_IDs))

# Display the DataFrame
df_UID_MID

100%|█████████▉| 279822/279838 [07:18<00:00, 662.27it/s]

224101 1775


,Uniprot ID,molecule ID,evidence,RHEA ID
0,A0A009IHW8,CHEBI:15377,exp,16301.0
1,A0A009IHW8,CHEBI:57540,exp,16301.0
2,A0A022PMU5,C00030,phylo,NaN
3,A0A022PN36,CHEBI:57318,phylo,22432.0
4,A0A022PN36,CHEBI:57540,phylo,22432.0
...,...,...,...,...
479802,Z4YNJ9,CHEBI:15378,phylo,19721.0
479803,Z4YNJ9,CHEBI:15379,phylo,19721.0
479804,Z4YNJ9,CHEBI:57394,phylo,19721.0
479805,Z4YNJ9,C00154,phylo,NaN


100%|██████████| 279838/279838 [07:29<00:00, 662.27it/s]

In [39]:
len(df_UID_MID), len(df_UID_MID.loc[df_UID_MID["evidence"] == "exp"])

(464469, 29376)

In [40]:
'''#magnesium, Phosphoprotein,  Collagen, C02, O2, Calmodulin, water, RNA, calcium, iron, hydrogen,
protein polypeptide chain,DNA, ammonium, lipopolysaccharide, fatty acid anion, double stranded DNA,
iron, hydrogen donor, hydrogen acceptor, lipid''';

remove_mets = ["CHEBI:18420", "C00562", "C00211", "CHEBI:16526", "CHEBI:15379", "C00391", "CHEBI:15377",
               "C00046", "CHEBI:29108", "CHEBI:29034", "CHEBI:15378", "CHEBI:16541", "CHEBI:16991",
              "CHEBI:28938", "CHEBI:16412", "CHEBI:28868", "CHEBI:4705", "CHEBI:29033", "CHEBI:17499",
              "CHEBI:13193", "C01356"]

df_UID_MID = df_UID_MID.loc[~df_UID_MID["molecule ID"].isin(remove_mets)]
df_UID_MID

,Uniprot ID,molecule ID,evidence,RHEA ID
1,A0A009IHW8,CHEBI:57540,exp,16301.0
2,A0A022PMU5,C00030,phylo,NaN
3,A0A022PN36,CHEBI:57318,phylo,22432.0
4,A0A022PN36,CHEBI:57540,phylo,22432.0
5,A0A022PN56,C00005,phylo,NaN
...,...,...,...,...
479798,Z4YHZ5,CHEBI:16810,phylo,18945.0
479799,Z4YHZ5,CHEBI:50342,phylo,18945.0
479804,Z4YNJ9,CHEBI:57394,phylo,19721.0
479805,Z4YNJ9,C00154,phylo,NaN


In [41]:
df_exp = df_UID_MID.loc[df_UID_MID["evidence"] == "exp"]
df_phylo = df_UID_MID.loc[df_UID_MID["evidence"] == "phylo"]

print("We have %s entries with phylogenetic evidence and %s entries with experimental evidence" % (len(df_phylo), len(df_exp)))

print("\n experimental dataset:")
print("Number of different enzymes: %s, Number of different substrates: %s"
      % (len(set(df_exp["Uniprot ID"])), len(set(df_exp["molecule ID"]))) )

print("\n phylogenetic dataset:")
print("Number of different enzymes: %s, Number of different substrates: %s"
      % (len(set(df_phylo["Uniprot ID"])), len(set(df_phylo["molecule ID"]))) )

We have 343043 entries with phylogenetic evidence and 23277 entries with experimental evidence

 experimental dataset:
Number of different enzymes: 13164, Number of different substrates: 1758

 phylogenetic dataset:
Number of different enzymes: 216606, Number of different substrates: 894


In [42]:
df_UID_MID.to_csv(os.path.join(CURRENT_DIR, "data", "UID_MID.csv"), sep = ",", index = False)

In [3]:
import pandas as pd
import os

df_UID_MID = pd.read_csv(os.path.join("data", "UID_MID.csv"), sep = ",")

In [5]:
df_UID_MID["Uniprot ID"].unique().shape

(222893,)

In [6]:
df_UID_MID

,Uniprot ID,molecule ID,evidence,RHEA ID
0,A0A009IHW8,CHEBI:57540,exp,16301.0
1,A0A022PMU5,C00030,phylo,NaN
2,A0A022PN36,CHEBI:57318,phylo,22432.0
3,A0A022PN36,CHEBI:57540,phylo,22432.0
4,A0A022PN56,C00005,phylo,NaN
...,...,...,...,...
366315,Z4YHZ5,CHEBI:16810,phylo,18945.0
366316,Z4YHZ5,CHEBI:50342,phylo,18945.0
366317,Z4YNJ9,CHEBI:57394,phylo,19721.0
366318,Z4YNJ9,C00154,phylo,NaN


In [7]:
swiss_prot = pd.read_csv("swiss_prot_enzymes.csv")

In [54]:
swiss_prot_ = swiss_prot[swiss_prot["accession"].isin(df_UID_MID["Uniprot ID"])]

In [9]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

records = []
for index, row in swiss_prot.iterrows():
    sequence = row["sequence"]
    header = row["accession"]
    record = SeqRecord(Seq(sequence), id=header, description="")
    records.append(record)

with open(os.path.join("data", "all_sequences.fasta"), 'w') as output_handle:
    SeqIO.write(records, output_handle, "fasta")